# 02 -- Build 1-min feature dataset

Canonical 1-min-cadence pipeline. Reads `data/raw_data/klines_1m.parquet`, runs the feature engine at `label_cadence='1min'`, and writes the model dataset along with its feature list and metadata sidecar.

Outputs:
- `data/model_dataset/dataset_1min.parquet`
- `data/model_dataset/feature_list_1min.json`
- `data/model_dataset/dataset_metadata_1min.json`

This notebook is the only active feature-building entrypoint. The pre-refactor boundary-cadence pipeline is archived under `legacy/` and produces no active artifacts.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
import warnings
from pathlib import Path

import polars as pl
import pandas as pd

# Resolve repo root robustly (works whether you launch Jupyter from repo root or notebooks/).
ROOT = Path.cwd()
if not (ROOT / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
    if (ROOT.parent / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError('Could not locate repo root.')
sys.path.insert(0, str(ROOT))

from src.analytics.audits import causal_feature_audit
from src.features.config import C, ETA, K_WARMUP, M, N_WARMUP, PHI
from src.features.pipeline import (
    _BASE_COLS,
    _DERIV_BASE_COLS,
    _LABEL_AUX_COLS,
    _RAW_COLS,
    run_pipeline,
)
from src.utils import get_git_sha

warnings.filterwarnings('ignore')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')

# Default target/execution alignment: at 1-min cadence the production
# strategy fills a long TP on intrabar HIGH crossing, so the label tests
# future highs (default in run_pipeline). The build records the source
# used so a model train + strategy backtest cannot silently drift apart.
# Flip to "close" for the legacy close-confirmed variant.
BARRIER_SOURCE = "high"

RAW_PATH = ROOT / 'data' / 'raw_data' / 'klines_1m.parquet'
OUT_DIR = ROOT / 'data' / 'model_dataset'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_PATH:', RAW_PATH)
print('OUT_DIR:', OUT_DIR)
print('BARRIER_SOURCE:', BARRIER_SOURCE)


ROOT: C:\Users\vitil\OneDrive\Desktop\barrier_classifier
RAW_PATH: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\raw_data\klines_1m.parquet
OUT_DIR: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset
BARRIER_SOURCE: high


## 1. Load raw 1-min klines

In [2]:
raw = pd.read_parquet(RAW_PATH)
print(f"Raw bars: {len(raw):,} rows  range {raw.index.min()} -> {raw.index.max()}")


Raw bars: 525,600 rows  range 2025-01-01 00:01:00+00:00 -> 2026-01-01 00:00:00+00:00


## 2. Run the 1-min feature pipeline

Calls `run_pipeline(label_cadence='1min', barrier_source=BARRIER_SOURCE, with_derivatives=False)`. The 1-min cadence emits a label on every row (no boundary sparsity), tests intrabar highs by default to match the production strategy's TP fill on `high >= entry * (1 + PHI)`, and produces the canonical feature set without derivatives base series.

In [3]:
print(
    f"Building dataset at label_cadence='1min', barrier_source='{BARRIER_SOURCE}'..."
)
t0 = time.perf_counter()
ds_1min = run_pipeline(
    raw,
    with_derivatives=False,
    label_cadence="1min",
    barrier_source=BARRIER_SOURCE,
)
dt = time.perf_counter() - t0
print(f"  ran in {dt:.1f}s")
print(f"  rows={len(ds_1min):,}  cols={len(ds_1min.columns)}")
print(f"  base_rate y = {float(ds_1min['y'].mean()):.4f}")
print(f"  ts range: {ds_1min['ts'].min()} -> {ds_1min['ts'].max()}")


Building dataset at label_cadence='1min', barrier_source='high'...


  ran in 34.5s
  rows=505,421  cols=1313
  base_rate y = 0.2100
  ts range: 2025-01-15 00:00:00 -> 2025-12-31 23:40:00


## 3. Sanity checks

Counts autocorr / rolling-excursion columns and verifies no boundary-sparse excursion columns leaked through (hard correctness gate).

In [4]:
n_autocorr = sum(1 for c in ds_1min.columns if c.startswith("target__autocorr_"))
print(f"  autocorr cols: {n_autocorr}")
n_roll_excursion = sum(
    1 for c in ds_1min.columns
    if c.startswith("excursion__roll_max_drawup__f__")
    or c.startswith("excursion__roll_max_drawdown__f__")
)
print(f"  rolling-excursion cols (every-row trailing): {n_roll_excursion}")
n_sparse_excursion = sum(
    1 for c in ds_1min.columns
    if c.startswith("excursion__max_drawup__f__")
    or c.startswith("excursion__max_drawdown__f__")
)
# Runtime check (not a Python ``assert``): under ``python -O`` asserts
# are stripped, but this leak invariant is a hard correctness gate.
if n_sparse_excursion != 0:
    raise RuntimeError(
        f"Boundary-sparse excursion cols leaked into the 1-min dataset: "
        f"{n_sparse_excursion}. run_pipeline(label_cadence='1min') should "
        "have dropped these after the engine ran -- check pipeline.py."
    )
print(f"  boundary-sparse excursion cols (must be 0): {n_sparse_excursion}")


  autocorr cols: 16
  rolling-excursion cols (every-row trailing): 18
  boundary-sparse excursion cols (must be 0): 0


## 4. Asymmetric loss weighting (barrier-distance)

Materialize the legacy barrier-distance weight on the training-eligible rows.
Near-miss negatives (`m_k` just below `phi`) are exponentially upweighted so the
classifier learns the geometry of the decision boundary. Positives keep
`weight = 1` (`use_pos=False`); time discount is disabled. The weight is stored
on the dataset so `03_train_model` can pass it to the CatBoost Pool, and is
explicitly excluded from `feature_list` in the next section.

In [5]:
import numpy as np

from src.utils import (
    WEIGHT_DIST_Q_TAIL,
    WEIGHT_DIST_W_MAX,
    compute_training_weights,
)

m_k = ds_1min["m_k"].to_numpy()
phi_arr = ds_1min["phi"].to_numpy()
phi_const = float(phi_arr[0])
if not np.allclose(phi_arr, phi_const):
    raise RuntimeError(
        f"phi is not constant across the dataset; range "
        f"[{float(np.nanmin(phi_arr))}, {float(np.nanmax(phi_arr))}]. "
        "Asymmetric weights assume a single phi."
    )

w_combined, w_dist, w_time, w_info = compute_training_weights(
    m_k=m_k,
    phi=phi_const,
    use_dist=True,
    use_time=False,
    w_max=WEIGHT_DIST_W_MAX,
    q_tail=WEIGHT_DIST_Q_TAIL,
    use_pos=False,
    normalize=False,
)

ds_1min = ds_1min.with_columns(
    pl.Series("weight", w_combined.astype(np.float64))
)

dist_info = w_info["barrier_distance"]
combined_info = w_info["combined"]
print("Asymmetric loss weighting (barrier-distance):")
print(f"  use_dist     : True")
print(f"  use_time     : False")
print(f"  use_pos      : False  (positives keep w=1)")
print(f"  w_max (neg)  : {WEIGHT_DIST_W_MAX}")
print(f"  q_tail (neg) : {WEIGHT_DIST_Q_TAIL}")
print(f"  d_star (neg) : {dist_info['d_star']:.6f}")
print(f"  lambda (neg) : {dist_info['lambda']:.4f}")
print(f"  n positives  : {dist_info['n_positive']:,}")
print(f"  n negatives  : {dist_info['n_negative']:,}")
print(f"  weight range : [{float(w_combined.min()):.4f}, {float(w_combined.max()):.4f}]")
print(f"  weight mean  : {float(w_combined.mean()):.4f}")
print(f"  effective n  : {combined_info['effective_n']:,.1f}  (raw n={len(w_combined):,})")


Asymmetric loss weighting (barrier-distance):
  use_dist     : True
  use_time     : False
  use_pos      : False  (positives keep w=1)
  w_max (neg)  : 5.0
  q_tail (neg) : 0.001
  d_star (neg) : 0.002500
  lambda (neg) : 643.7446
  n positives  : 106,149
  n negatives  : 399,272
  weight range : [1.0000, 5.0000]
  weight mean  : 2.6381
  effective n  : 397,026.9  (raw n=505,421)


## 5. Causal feature audit

Derives the feature list by excluding label aux, raw OHLCV, base series, derivatives base, the asymmetric `weight` column from Section 4, and `undef__*` columns -- reusing the pipeline's authoritative constants so the exclusion set cannot drift from the imputation-step contract. `causal_feature_audit` is a static naming check requiring every feature to contain `__f__` or `__h__` and forbidding suspect tokens like `fwd`, `future`, `ahead`.

In [6]:
# Feature list: every column that is not a label, raw OHLCV, base
# series, or derivatives base series. Reuses the pipeline's
# authoritative constants so the exclusion set cannot drift from the
# imputation-step contract. Triple-barrier aux columns (m_dn, tau_dn)
# are included in _LABEL_AUX_COLS -- sidecar diagnostics, not features.
# The asymmetric `weight` column (Section 4) is consumed by 03_train_model
# via the CatBoost Pool and must not leak into the feature matrix.
NON_FEATURE = (
    set(_LABEL_AUX_COLS)
    | set(_RAW_COLS)
    | set(_BASE_COLS)
    | set(_DERIV_BASE_COLS)
    | {"weight"}
)
feature_cols = [
    c for c in ds_1min.columns
    if c not in NON_FEATURE and not c.startswith("undef__")
]
# Static causal-naming audit: every feature must contain ``__f__`` or
# ``__h__`` and no suspect tokens (``fwd``, ``future``, ``ahead``, ...).
audit = causal_feature_audit(feature_cols)
if not audit.passed:
    raise RuntimeError(
        "causal_feature_audit failed on 1-min build:\n"
        f"  suspect = {audit.suspect[:10]}\n"
        f"  unmatched = {audit.unmatched[:10]}"
    )
if "weight" in feature_cols:
    raise RuntimeError(
        "Asymmetric `weight` column leaked into feature_cols; "
        "fix the NON_FEATURE exclusion set above."
    )
print(f"  causal audit passed: {audit.n_causal}/{audit.n_features} causal")


  causal audit passed: 1217/1217 causal


## 6. Save artifacts

Writes the parquet dataset (including the asymmetric `weight` column), the feature list JSON, and the metadata sidecar (including the current `git_sha` for provenance).

In [7]:
ds_path = OUT_DIR / 'dataset_1min.parquet'
ds_1min.write_parquet(str(ds_path))
print(f"  wrote {ds_path}")

feature_list_path = OUT_DIR / 'feature_list_1min.json'
with open(feature_list_path, "w") as f:
    json.dump(feature_cols, f, indent=2)
print(f"  feature_list_1min.json: {len(feature_cols)} features")

metadata = {
    "label_cadence": "1min",
    "barrier_source": BARRIER_SOURCE,
    "M": int(M),
    "N_WARMUP": int(N_WARMUP),
    "K_WARMUP": int(K_WARMUP),
    "PHI": float(PHI),
    "C": float(C),
    "ETA": float(ETA),
    "n_rows": int(len(ds_1min)),
    "n_features": len(feature_cols),
    "ts_start": str(ds_1min["ts"].min()),
    "ts_end": str(ds_1min["ts"].max()),
    "base_rate": float(ds_1min["y"].mean()),
    "build_time_seconds": dt,
    "git_sha": get_git_sha(),
    "weighting": {
        "scheme": "barrier_distance_asymmetric",
        "use_dist": True,
        "use_time": False,
        "use_pos": False,
        "w_max_neg": float(WEIGHT_DIST_W_MAX),
        "q_tail_neg": float(WEIGHT_DIST_Q_TAIL),
        "weight_min": float(w_combined.min()),
        "weight_max": float(w_combined.max()),
        "weight_mean": float(w_combined.mean()),
        "effective_n": float(combined_info["effective_n"]),
    },
}
metadata_path = OUT_DIR / 'dataset_metadata_1min.json'
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  metadata: {metadata}")


  wrote C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\dataset_1min.parquet
  feature_list_1min.json: 1217 features
  metadata: {'label_cadence': '1min', 'barrier_source': 'high', 'M': 20, 'N_WARMUP': 20159, 'K_WARMUP': 1008, 'PHI': 0.0025, 'C': 0.0023, 'ETA': 0.0002, 'n_rows': 505421, 'n_features': 1217, 'ts_start': '2025-01-15 00:00:00', 'ts_end': '2025-12-31 23:40:00', 'base_rate': 0.2100209528294234, 'build_time_seconds': 34.45507389999693, 'git_sha': '3c0fc03a5e15ff5d18c5bea6d2c5545e18a2c4ba', 'weighting': {'scheme': 'barrier_distance_asymmetric', 'use_dist': True, 'use_time': False, 'use_pos': False, 'w_max_neg': 5.0, 'q_tail_neg': 0.001, 'weight_min': 1.0, 'weight_max': 4.999999999999999, 'weight_mean': 2.6381032409779084, 'effective_n': 397026.8614957811}}


## 7. Summary

In [8]:
summary = {
    "n_rows": int(len(ds_1min)),
    "n_features": len(feature_cols),
    "base_rate": float(ds_1min["y"].mean()),
    "ts_start": str(ds_1min["ts"].min()),
    "ts_end": str(ds_1min["ts"].max()),
    "barrier_source": BARRIER_SOURCE,
    "build_time_seconds": round(dt, 2),
}
print(summary)


{'n_rows': 505421, 'n_features': 1217, 'base_rate': 0.2100209528294234, 'ts_start': '2025-01-15 00:00:00', 'ts_end': '2025-12-31 23:40:00', 'barrier_source': 'high', 'build_time_seconds': 34.46}
